# 16｜序列模型_RNN與時間序列預測

把連續時間序列切成視窗，嚴格使用過去預測下一步。

從上到下執行程式格。第一格安裝缺少的套件，第二格準備本章所需設定。

## 投影片與程式的閱讀方式

對應 `slides/16_序列模型_RNN與時間序列預測.md`，目前 36 頁。逐圖完整表見 `programs/SLIDE_NOTEBOOK_MAP_05_25.md`。

RNN/BPTT、固定起點多步預測、LSTM/GRU 與擴張因果卷積。

各格會標示實驗資料、評估方式與適用範圍；架構圖用於說明機制與張量形狀。

| 程式代碼 | 投影片頁次 | 本格學習內容 |
|---|---|---|
| 16-03 | 10、11、13、14、15 | 課堂小實驗 |
| 16-04 | 3、10、11、15 | 執行結果圖 |
| 16-05 | 4、5、6、7、8、9、16、17、24、36 | 1. 狀態、展開、序列介面與 BPTT |
| 16-06 | 10、11、12、13、14、15、18、19、20、21、22、23、33、36 | 2. 季節基準、多步預測與標籤對齊 |
| 16-07 | 24、25、26、27、28、29、30、31、32、34、35、36 | 3. LSTM、GRU 與擴張因果卷積 |

In [ ]:
# 第 1 格：安裝本 Notebook 實際使用的套件。
# 程式 16-01｜投影片 16：2 頁
import importlib.util, subprocess, sys
_needed = {'numpy': 'numpy>=1.26,<3', 'pandas': 'pandas>=2.0,<3', 'matplotlib': 'matplotlib>=3.7,<4', 'sklearn': 'scikit-learn>=1.4,<2'}
_missing = [spec for module, spec in _needed.items() if importlib.util.find_spec(module) is None]
if _missing:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', *_missing])

In [ ]:
# 第 2 格：本 Notebook 的輸出位置與繪圖設定。
# 程式 16-02｜投影片 16：2 頁
from pathlib import Path
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

SEED = 42
ROOT = Path.cwd()
if ROOT.name == "notebooks" and (ROOT.parent / "pyproject.toml").exists():
    ROOT = ROOT.parent
OUT = ROOT / ("outputs" if (ROOT / "pyproject.toml").exists() else "mlcourse_outputs")
plt.rcParams.update({"figure.figsize": (8, 4.5), "figure.dpi": 110,
                     "axes.grid": True, "grid.alpha": 0.2, "font.size": 11})

def savefig(name):
    folder = OUT / "figures"
    folder.mkdir(parents=True, exist_ok=True)
    plt.gcf().savefig(folder / f"{name}.png", dpi=150, bbox_inches="tight")
    svg_path = folder / f"{name}.svg"
    plt.gcf().savefig(svg_path, bbox_inches="tight", metadata={"Date": None})
    svg_path.write_text("\n".join(line.rstrip() for line in svg_path.read_text().splitlines()) + "\n")
    plt.show()
    plt.close(plt.gcf())

def report(name, values):
    folder = OUT / "reports"
    folder.mkdir(parents=True, exist_ok=True)
    (folder / f"{name}.json").write_text(
        json.dumps(values, ensure_ascii=False, indent=2, default=float) + "\n")

rng = np.random.default_rng(SEED)

## 課堂小實驗

這是逐步揭露真實觀測的 rolling one-step 評估：每個測試視窗可用當時已知的過去值。不是在單一時間點一次預測整段未來；下方另做固定起點多步預測。

In [ ]:
# 程式 16-03｜投影片 16：10、11、13、14、15 頁
from sklearn.linear_model import LinearRegression
t = np.arange(240)
series = np.sin(t / 12) + 0.35 * np.sin(t / 3.5) + rng.normal(0, 0.08, len(t))
window = 12
X = np.stack([series[i:i+window] for i in range(len(series)-window)])
y = series[window:]
split = 170
model = LinearRegression().fit(X[:split], y[:split])
pred = model.predict(X[split:])
persistence = X[split:, -1]
rmse = lambda a,b: float(np.sqrt(np.mean((a-b)**2)))
pd.Series({'window_model_RMSE': rmse(y[split:], pred), 'persistence_RMSE': rmse(y[split:], persistence)})

## 執行結果圖

In [ ]:
# 程式 16-04｜投影片 16：3、10、11、15 頁
idx = np.arange(split+window, len(series))
plt.plot(idx, y[split:], label='actual')
plt.plot(idx, pred, label='window model')
plt.plot(idx, persistence, alpha=.6, label='persistence')
plt.xlabel('time'); plt.ylabel('value'); plt.title('Forecast uses only the previous 12 values')
plt.legend()
savefig('16_sequence_demo')
report('sequence_forecast', {'window_RMSE': rmse(y[split:], pred), 'persistence_RMSE': rmse(y[split:], persistence)})

## 1. 狀態、展開、序列介面與 BPTT

同一組權重沿時間重用。用最後狀態的平方損失驗算共享循環權重的梯度，分支須把所有時間步的貢獻加總。

In [ ]:
# 程式 16-05｜投影片 16：4、5、6、7、8、9、16、17、24、36 頁
sequence16=np.array([1.,0.]);wx16=1.;wh16=.5
states16=[0.]
for value16 in sequence16:states16.append(np.tanh(wx16*value16+wh16*states16[-1]))
assert np.isclose(states16[1],.761594156) and np.isclose(states16[2],.363399484)
print('Hand states:',states16)
# d(0.5*h_T^2)/dWh：倒序走過展開圖
adj16=states16[-1];grad16=0.
for t16 in range(len(sequence16),0,-1):
    dz16=adj16*(1-states16[t16]**2);grad16+=dz16*states16[t16-1];adj16=dz16*wh16
loss16=lambda w:.5*np.tanh(w*np.tanh(1.))**2
eps16=1e-5;assert np.isclose(grad16,(loss16(wh16+eps16)-loss16(wh16-eps16))/(2*eps16),atol=1e-8)
rng16=np.random.default_rng(SEED);batch16=rng16.normal(size=(3,5,2));Wi16=rng16.normal(0,.3,(2,4));Wh16=rng16.normal(0,.2,(4,4))
h16=np.zeros((3,4));all16=[]
for t16 in range(5):h16=np.tanh(batch16[:,t16]@Wi16+h16@Wh16);all16.append(h16.copy())
all16=np.stack(all16,axis=1);Wo16=rng16.normal(size=(4,2))
print('seq-to-seq / seq-to-vector:',(all16@Wo16).shape,(h16@Wo16).shape)
# vector-to-seq / encoder-decoder：固定編碼作初始狀態，繼續展開
state16=h16.copy();generated16=[]
for _ in range(4):state16=np.tanh(state16@Wh16);generated16.append(state16@Wo16)
print('vector-to-seq:',np.stack(generated16,axis=1).shape)
# 上層 RNN 吃下層每一步的表示；時間深度與層深度不同
upper16=np.zeros((3,4));upper_history16=[]
for t16 in range(5):upper16=np.tanh(all16[:,t16]+upper16@Wh16);upper_history16.append(upper16.copy())
fig,axes=plt.subplots(1,2,figsize=(10,3.5));axes[0].plot(np.array(states16),'o-');axes[0].set(title='Shared-weight recurrence',xlabel='time',ylabel='h')
axes[1].imshow(all16[0].T,aspect='auto',cmap='coolwarm');axes[1].set(xlabel='time',ylabel='hidden unit',title='Hidden states across time');savefig('c16_recurrent_states')
print('BPTT gradient:',grad16)

## 2. 季節基準、多步預測與標籤對齊

固定起點的遞迴預測只能將模型自己的預測放回視窗；direct 模型需保證每個訓練目標在切分點以前。與逐步取得真實過去值的單步評估分開解讀。

In [ ]:
# 程式 16-06｜投影片 16：10、11、12、13、14、15、18、19、20、21、22、23、33、36 頁
H16=14;cut16=split+window
starts16=np.arange(len(series)-window-H16+1)
inputs16=np.stack([series[i:i+window] for i in starts16]);targets16=np.stack([series[i+window:i+window+H16] for i in starts16])
train16=starts16+window+H16<=cut16
direct16=LinearRegression().fit(inputs16[train16],targets16[train16])
origins16=np.arange(cut16,len(series)-H16+1,7);rec_predictions16=[];direct_predictions16=[];truth16=[];seasonal16=[]
for origin16 in origins16:
    context16=series[origin16-window:origin16].copy();future16=[]
    for horizon16 in range(H16):
        guess16=float(model.predict(context16[-window:][None])[0]);future16.append(guess16);context16=np.r_[context16,guess16]
    rec_predictions16.append(future16);direct_predictions16.append(direct16.predict(series[origin16-window:origin16][None])[0]);truth16.append(series[origin16:origin16+H16])
    seasonal16.append(np.resize(series[origin16-7:origin16],H16))
truth16=np.array(truth16);fig,axes=plt.subplots(1,2,figsize=(11,4))
for name16,p16 in [('recursive',np.array(rec_predictions16)),('direct',np.array(direct_predictions16)),('period-7 baseline',np.array(seasonal16))]:
    errors16=np.mean(np.abs(p16-truth16),axis=0);axes[0].plot(np.arange(1,H16+1),errors16,label=name16);axes[1].plot(np.arange(H16),p16[0],label=name16)
    print(name16,'MAE at horizons 1/7/14:',errors16[[0,6,13]])
axes[0].set(xlabel='horizon',ylabel='MAE across origins');axes[0].legend(fontsize=8)
axes[1].plot(truth16[0],'k--',label='truth');axes[1].legend(fontsize=8);axes[1].set_title('One fixed origin');savefig('c16_multistep_forecast')
print('Seq2seq target positions:',np.arange(3)[:,None]+np.arange(1,4)[None,:])
# AR / MA 機制示範：MA 使用創新誤差，不是移動平均平滑器
innov16=rng16.normal(0,.1,100);arma16=np.zeros(100)
for j16 in range(1,100):arma16[j16]=.6*arma16[j16-1]+innov16[j16]+.3*innov16[j16-1]
integrated16=np.cumsum(arma16);assert np.allclose(np.diff(integrated16),arma16[1:])
print('ARMA / integrated / seasonal difference shapes:',arma16.shape,integrated16.shape,(series[7:]-series[:-7]).shape)

## 3. LSTM、GRU 與擴張因果卷積

門控用逐元素公式驗算。卷積只左側補零；故意改未來值，檢查前面的輸出不變。GRU 採投影片候選更新慣例，框架重置門位置可能不同。

In [ ]:
# 程式 16-07｜投影片 16：24、25、26、27、28、29、30、31、32、34、35、36 頁
sig16=lambda z:1/(1+np.exp(-z))
cell16=.9*.8+.2*(-.5);hidden16=.5*np.tanh(cell16)
assert np.isclose(cell16,.62);print('LSTM cell / hidden:',cell16,hidden16)
state16=np.zeros(4);cell_state16=np.zeros(4);wxg16=rng16.normal(0,.2,(2,16));whg16=rng16.normal(0,.2,(4,16))
lstm_hist16=[]
for xt16 in batch16[0]:
    gates16=xt16@wxg16+state16@whg16;f16,i16,o16,candidate16=np.split(gates16,4)
    cell_state16=sig16(f16)*cell_state16+sig16(i16)*np.tanh(candidate16);state16=sig16(o16)*np.tanh(cell_state16);lstm_hist16.append(state16.copy())
hgru16=np.zeros(4);Wz16=rng16.normal(0,.2,(6,4));Wr16=rng16.normal(0,.2,(6,4));Wc16=rng16.normal(0,.2,(6,4));gru_hist16=[]
for xt16 in batch16[0]:
    z16=sig16(np.r_[xt16,hgru16]@Wz16);r16=sig16(np.r_[xt16,hgru16]@Wr16)
    candidate16=np.tanh(np.r_[xt16,r16*hgru16]@Wc16);hgru16=z16*hgru16+(1-z16)*candidate16;gru_hist16.append(hgru16.copy())
def causal16(x,kernel,dilation=1):
    padded=np.pad(x,((len(kernel)-1)*dilation,0));out=np.zeros_like(x,dtype=float)
    for j in range(len(kernel)):out+=kernel[j]*padded[j*dilation:j*dilation+len(x)]
    return out
signal16=np.arange(20,dtype=float);changed16=signal16.copy();changed16[10:]+=100
out16=signal16.copy();alt16=changed16.copy()
for d16 in [1,2,4]:out16=causal16(out16,np.ones(2)/2,d16);alt16=causal16(alt16,np.ones(2)/2,d16)
assert np.allclose(out16[:10],alt16[:10]);assert 1+sum([1,2,4])==8
fig,axes=plt.subplots(1,2,figsize=(10,3.5));axes[0].plot(lstm_hist16);axes[0].set_title('LSTM hidden dimensions')
axes[1].plot(out16,label='original future');axes[1].plot(alt16,'--',label='changed future');axes[1].axvline(10,c='gray');axes[1].legend();axes[1].set_title('Causal dilations 1,2,4');savefig('c16_gates_causality')
print('LSTM states (h,c) / GRU h:',state16.shape,cell_state16.shape,hgru16.shape)

## 練習：改動一個參數

先預測結果，再修改程式中的一個參數並重跑；比較圖表或數值，說明差異。